In [1]:
import librosa
import numpy as np
import logging
import hashlib
import matplotlib.pyplot as plt
from moviepy import VideoFileClip
import os

# --- Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - [AUDIT] - %(message)s',
    handlers=[logging.FileHandler("sync_audit.log"), logging.StreamHandler()]
)

def get_file_hash(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

def extract_audio(video_path):
    output_path = video_path.rsplit('.', 1)[0] + ".wav"
    if not os.path.exists(output_path):
        clip = VideoFileClip(video_path)
        clip.audio.write_audiofile(output_path, logger=None)
    return output_path

def calculate_metrics(ref_audio, target_audio, sr=16000):
    ref_norm = (ref_audio - np.mean(ref_audio)) / (np.std(ref_audio) + 1e-9)
    tar_norm = (target_audio - np.mean(target_audio)) / (np.std(target_audio) + 1e-9)
    corr = np.correlate(ref_norm, tar_norm, mode='full') / len(ref_norm)
    peak_val = np.max(corr)
    lag = np.argmax(corr) - (len(tar_norm) - 1)
    return peak_val, lag / sr, corr


def plot_and_save(corr, sr, filename):
    """Visualizes the synchronization peak with corrected lag array dimensions."""
    # The length of a 'full' correlation is len(y1) + len(y2) - 1
    # We create a range centered at 0
    n = len(corr)
    lags = np.arange(-n // 2, n // 2 + (n % 2)) / sr
    
    plt.figure(figsize=(10, 4))
    # Ensure dimensions match by slicing if necessary, 
    # though the math above should align them perfectly
    plt.plot(lags[:len(corr)], corr) 
    
    plt.title(f"Sync Alignment: {filename}")
    plt.xlabel("Time Lag (s)")
    plt.ylabel("Normalized Correlation")
    plt.axvline(x=0, color='r', linestyle='--')
    plt.grid(True)
    
    # Clean the filename for saving
    safe_name = os.path.basename(filename).split('.')[0]
    plt.savefig(f"sync_plot_{safe_name}.png")
    plt.close()


def plot_waveforms_with_offset_lines(ref_audio, target_audio, lag_samples, filename):
    # Ensure directory exists
    os.makedirs("sync_plots", exist_ok=True)
    
    # Slice a window of 2000 samples to zoom into the alignment event
    # We choose the middle of the signals for this check
    mid = len(ref_audio) // 2
    window = 2000
    
    ref_slice = ref_audio[mid : mid + window]
    # Target is shifted by the lag_samples calculated earlier
    target_slice = target_audio[mid + int(lag_samples) : mid + int(lag_samples) + window]

    plt.figure(figsize=(12, 5))
    
    # Plot waveforms
    plt.plot(ref_slice, label="Reference Audio", alpha=0.7, color='blue')
    plt.plot(target_slice, label="Target Audio (Shifted)", alpha=0.7, color='orange')
    
    # Draw two vertical lines to show the alignment anchor points
    # In a perfect sync, these two lines will be at the exact same x-coordinate
    plt.axvline(x=window // 2, color='blue', linestyle='--', label="Ref Anchor")
    plt.axvline(x=window // 2 + (lag_samples % 1), color='orange', linestyle='--', label="Target Anchor")
    
    plt.title(f"Visual Offset Audit: {os.path.basename(filename)}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"sync_plots/offset_{os.path.basename(filename).split('.')[0]}.png")
    plt.close()

def plot_all_waveforms_together(ref_audio, targets_dict, sr=16000):
    """
    Plots the reference audio and all target audios on one graph.
    Legend only shows filenames.
    """
    os.makedirs("sync_plots", exist_ok=True)
    plt.figure(figsize=(15, 6))
    
    mid = len(ref_audio) // 2
    window = 2000
    
    # Plot Reference
    plt.plot(ref_audio[mid : mid + window], label="Reference", color='black', linewidth=2, alpha=0.8)
    
    # Plot each Target
    # We use a color map for variety
    colors = plt.cm.tab10(np.linspace(0, 1, len(targets_dict)))
    
    for i, (name, (audio, lag_sec)) in enumerate(targets_dict.items()):
        shift = int(lag_sec * sr)
        target_slice = audio[mid + shift : mid + shift + window]
        
        # Strip path for clean label
        label_name = os.path.basename(name)
        plt.plot(target_slice, label=label_name, color=colors[i], alpha=0.7)
    
    plt.title("Multi-Channel Audio Alignment Audit")
    plt.xlabel("Sample Window")
    plt.ylabel("Normalized Amplitude")
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.3)
    
    plt.savefig("sync_plots/all_waveforms_overlap.png")
    plt.close()


def process_stadium_sync(files):
    ref_file = extract_audio(files[0])
    y_ref, sr = librosa.load(ref_file, sr=16000)
    
    # Store results here for the multi-plot
    targets_data = {} 
    
    for i in range(1, len(files)):
        target_path = extract_audio(files[i])
        y_tar, _ = librosa.load(target_path, sr=16000)
        
        # 1. Calculate Metrics
        peak, lag_sec, corr = calculate_metrics(y_ref, y_tar)
        
        # 2. Calculate Jitter (Stability across 5 segments)
        # Using the logic to measure standard deviation of offsets
        segments = 5
        seg_len = len(y_ref) // segments
        offsets = []
        for s in range(segments):
            sub_ref = y_ref[s*seg_len:(s+1)*seg_len]
            sub_tar = y_tar[s*seg_len:(s+1)*seg_len]
            # Use cross-correlation to find segment-specific lag
            sub_corr = np.correlate(sub_ref, sub_tar, mode='full')
            offsets.append(np.argmax(sub_corr) - (len(sub_tar) - 1))
        
        jitter = np.std(offsets) / sr
        
        # 3. Save individual plots
        plot_and_save(corr, sr, files[i])
        plot_waveforms_with_offset_lines(y_ref, y_tar, lag_sec * sr, files[i])
        
        # 4. Collect data for the big plot (store jitter if needed later)
        targets_data[files[i]] = (y_tar, lag_sec)
        
        # 5. Updated Logging to include Jitter
        status = "PASS" if peak > 0.3 else "FAIL"
        logging.info(f"[{status}] Pair: {files[0]} <-> {files[i]} | Peak: {peak:.4f} | Lag: {lag_sec:.4f}s | Jitter: {jitter:.4f}s")
    
    # 6. Final Visualization
    plot_all_waveforms_together(y_ref, targets_data)


In [2]:
# Execute
files = ["v-data/AN01.mp4", "v-data/AN02.mp4", "v-data/AN03.mp4", "v-data/AN04.mp4"]
process_stadium_sync(files)

/data/hirusha/test_synco_volley/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-05 22:11:18,379 - INFO - [AUDIT] - [PASS] Pair: v-data/AN01.mp4 <-> v-data/AN02.mp4 | Peak: 0.3196 | Lag: 0.1276s | Jitter: 0.0949s
2026-06-05 22:11:42,073 - INFO - [AUDIT] - [PASS] Pair: v-data/AN01.mp4 <-> v-data/AN03.mp4 | Peak: 0.5177 | Lag: 0.1146s | Jitter: 0.0001s
2026-06-05 22:12:05,959 - INFO - [AUDIT] - [PASS] Pair: v-data/AN01.mp4 <-> v-data/AN04.mp4 | Peak: 0.4082 | Lag: -0.0793s | Jitter: 0.0003s


In [ ]:
# Execute
# add manually annotated video file path
files = ["v-data/AN02.mp4", "v-data/AN01.mp4", "v-data/AN03.mp4", "v-data/AN04.mp4"]
process_stadium_sync(files)

2026-06-05 22:12:30,135 - INFO - [AUDIT] - [PASS] Pair: v-data/AN02.mp4 <-> v-data/AN01.mp4 | Peak: 0.3196 | Lag: -0.1276s | Jitter: 0.0949s
2026-06-05 22:12:53,776 - INFO - [AUDIT] - [PASS] Pair: v-data/AN02.mp4 <-> v-data/AN03.mp4 | Peak: 0.5202 | Lag: -0.0130s | Jitter: 0.0002s
2026-06-05 22:13:17,431 - INFO - [AUDIT] - [PASS] Pair: v-data/AN02.mp4 <-> v-data/AN04.mp4 | Peak: 0.3058 | Lag: 0.0109s | Jitter: 0.0005s


In [4]:
# Execute
files = ["v-data/AN03.mp4", "v-data/AN01.mp4", "v-data/AN02.mp4", "v-data/AN04.mp4"]
process_stadium_sync(files)

2026-06-05 22:13:41,378 - INFO - [AUDIT] - [PASS] Pair: v-data/AN03.mp4 <-> v-data/AN01.mp4 | Peak: 0.5177 | Lag: -0.1146s | Jitter: 0.0001s
2026-06-05 22:14:04,948 - INFO - [AUDIT] - [PASS] Pair: v-data/AN03.mp4 <-> v-data/AN02.mp4 | Peak: 0.5202 | Lag: 0.0130s | Jitter: 0.0002s
2026-06-05 22:14:28,747 - INFO - [AUDIT] - [PASS] Pair: v-data/AN03.mp4 <-> v-data/AN04.mp4 | Peak: 0.5232 | Lag: -0.1269s | Jitter: 0.0002s


In [5]:
# Execute
files = ["v-data/AN04.mp4", "v-data/AN01.mp4", "v-data/AN02.mp4", "v-data/AN03.mp4"]
process_stadium_sync(files)

2026-06-05 22:14:52,755 - INFO - [AUDIT] - [PASS] Pair: v-data/AN04.mp4 <-> v-data/AN01.mp4 | Peak: 0.4082 | Lag: 0.0793s | Jitter: 0.0003s
2026-06-05 22:15:16,437 - INFO - [AUDIT] - [PASS] Pair: v-data/AN04.mp4 <-> v-data/AN02.mp4 | Peak: 0.3058 | Lag: -0.0109s | Jitter: 0.0005s
2026-06-05 22:15:40,363 - INFO - [AUDIT] - [PASS] Pair: v-data/AN04.mp4 <-> v-data/AN03.mp4 | Peak: 0.5232 | Lag: 0.1269s | Jitter: 0.0002s


In [6]:
import re

def generate_summary(log_file):
    lags, peaks, jitters = [], [], []
    
    with open(log_file, 'r') as f:
        for line in f:
            if "[PASS]" in line or "[FAIL]" in line:
                # Regex to extract numeric values
                p = re.findall(r"Peak: ([\d\.]+)", line)
                l = re.findall(r"Lag: (-?[\d\.]+)s", line)
                j = re.findall(r"Jitter: ([\d\.]+)s", line)
                
                if p and l and j:
                    peaks.append(float(p[0]))
                    lags.append(float(l[0]))
                    jitters.append(float(j[0]))
    
    print("--- Synchronization Accuracy Report ---")
    print(f"Average Peak Similarity: {np.mean(peaks):.5f}")
    print(f"Mean Absolute Lag:       {np.mean(np.abs(lags)):.5f} seconds")
    print(f"Max Recorded Lag:        {np.max(np.abs(lags)):.5f} seconds")
    print(f"Avg Jitter (Stability):  {np.mean(jitters):.5f} seconds")

generate_summary("sync_audit.log")

--- Synchronization Accuracy Report ---
Average Peak Similarity: 0.43245
Mean Absolute Lag:       0.07872 seconds
Max Recorded Lag:        0.12760 seconds
Avg Jitter (Stability):  0.01603 seconds
